# Classification: your DataFrame, a reusable predictor

SDK 0.7. Download this notebook and its matching support.py into the same directory. Restart/run all. Default mode is controlled synthetic HTTP transport, not a foundation model or scientific benchmark. Explicit live mode makes a hosted fit/predict and retains its synthetic state. No fallback, deletion or saved prediction output. The frozen recipe is seed9, 128 training rows, four features and 32 query rows. Read README.md before live use.


## Build separate training and query tables

The training table includes the explicit target. Query rows include features only. The helper provides the documented frozen recipe and keeps fixture plumbing out of the public estimator API.


In [ ]:
from support import estimator_transport, synthetic_table
import pandas as pd
import numpy as np
from welt import Classifier

columns, values, target, query_values = synthetic_table()
train = pd.DataFrame(values, columns=columns)
train["label"] = target
query = pd.DataFrame(query_values, columns=columns)


## Fit, predict and reopen

Choose `Classifier` explicitly. `fit(train, target="label")` excludes the named target from features. Fit uploads and waits for remote preparation; prediction executes remotely. In default mode the context below installs only the controlled transport. In live mode the same public SDK calls run against the configured service.


In [ ]:
with estimator_transport():
    model = Classifier(model="tabicl-v2", random_state=9)
    model.fit(train, target="label")
    predictions = model.predict(query)
    assert predictions.shape == (32,)
    pinned_version = model.model_version_
    reopened = Classifier.from_predictor(model.predictor_id_)
    repeat = reopened.predict(query[columns[::-1]])
    assert reopened.model_version_ == pinned_version
    np.testing.assert_array_equal(predictions, repeat)
    probabilities = model.predict_proba(query)
    assert probabilities.shape == (32, len(model.classes_))
    assert np.allclose(probabilities.sum(axis=1), 1, atol=1e-5)
print(type(predictions).__name__, predictions.shape)


## Read the result

Expected output contract: `ndarray (32,)`, one result per query row in order. Classification probabilities use `classes_` order and are not calibrated confidence. Regression returns finite mean points in target units, without class probabilities or calibrated intervals. Reopening preserves task/model/configuration/version/seed and makes no new fit or training upload; this does not force a worker restart. Keep `predictor_id_` as experiment metadata. Local timeout preserves durable work: reconnect by job ID instead of refitting. Sources contain no execution results.
